# Gender-Based Name Classification using Machine Learning and Docker Deployment




## Problem Statement

Accurately predicting a person's gender from their first name has practical applications in personalization engines, CRM systems, and demographic analytics. Traditional rule-based or dictionary lookup approaches fail to generalize across diverse, multicultural name datasets.

The objective of this project is to train a deep learning model that classifies gender (male / female) from a given name using contextual language understanding — and then serve it as a containerized, production-ready API.

---

## Project Summary

- **Dataset:** `users.csv` containing `name` and `gender` columns. Gender is binarized: `male → 0`, `female → 1`.
- **Tokenization:** Names are tokenized using the **BERT tokenizer** (`bert-base-uncased`) with padding and truncation applied uniformly.
- **Model:** A **BERT for Sequence Classification** model (`bert-base-uncased`, 2 labels) is fine-tuned over 3 epochs using the **AdamW optimizer** (lr = 2e-5) on an 80/20 train-validation split with batch size 16.
- **Evaluation:** Validation accuracy is reported post-training; the model and tokenizer are saved to `bert_gender_classifier/` for deployment.
- **Production Deployment (Part 2):** A Flask REST API wraps the fine-tuned BERT model with lazy model loading, full input validation (HTTP 400 on bad input), and a confidence score alongside the predicted gender. The API is containerized with **Docker**, deployed on **Kubernetes**, tracked with **MLflow** (SQLite backend), and includes a **Jenkins** CI/CD pipeline. The API is demonstrated live within the notebook.

Git hub link: https://github.com/umerulla/Travel-Tourism-ML-System

In [ ]:
#Upload and Extract ZIP File

# Step 1: Upload your ZIP file
from google.colab import files
uploaded = files.upload()


Saving travel_capstone.zip to travel_capstone.zip


In [ ]:
# Step 2: Extract the contents of the uploaded ZIP file
import zipfile
import os

uploaded_zip = list(uploaded.keys())[0]

with zipfile.ZipFile(uploaded_zip, 'r') as zip_ref:
    zip_ref.extractall("name_gender_data")


✅ ZIP extracted to /content/gender_data


In [ ]:
print("✅ ZIP file successfully extracted to /content/name_gender_data")


✅ ZIP extracted to /content/gender_data


In [ ]:
# Step 3: Display all extracted files
for root, dirs, files in os.walk("name_gender_data"):
    for filename in files:
        print(filename)


users.csv
hotels.csv
flights.csv


In [ ]:
# Load and Preview the Dataset
# Step 4: Load the CSV dataset
import pandas as pd

# Load users.csv from the extracted folder
df = pd.read_csv("name_gender_data/users.csv")

# Preview the first few rows
df.head()


,code,company,name,gender,age
0,0,4You,Roy Braun,male,21
1,1,4You,Joseph Holsten,male,37
2,2,4You,Wilma Mcinnis,female,48
3,3,4You,Paula Daniel,female,23
4,4,4You,Patricia Carson,female,44


In [ ]:
#Preprocess the Dataset
# Step 3: Clean and encode the dataset

# Keep only 'name' and 'gender' columns
df = df[['name', 'gender']]

# Remove rows where gender is not 'male' or 'female'
df = df[df['gender'].isin(['male', 'female'])].copy()

# Encode gender: male → 0, female → 1
df['label'] = df['gender'].map({'male': 0, 'female': 1}).astype(int)

# Reset index
df.reset_index(drop=True, inplace=True)

# Check distribution
print("Label distribution in the dataset:\n", df['label'].value_counts())
df.head()


Class distribution:
 label
0    452
1    448
Name: count, dtype: int64


,name,gender,label
0,Roy Braun,male,0
1,Joseph Holsten,male,0
2,Wilma Mcinnis,female,1
3,Paula Daniel,female,1
4,Patricia Carson,female,1


In [ ]:
#Tokenize Names Using BERT Tokenizer
# Install Hugging Face Transformers if not already installed
!pip install transformers --quiet


In [ ]:
# Step 4b: Tokenize the names using BERT tokenizer
from transformers import BertTokenizer

# Load the pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize all names in the dataset
tokens = tokenizer(
    df['name'].tolist(),
    padding=True,
    truncation=True,
    return_tensors='pt'  # Return PyTorch tensors
)

# Check tokenized shapes
print("Input IDs shape:", tokens['input_ids'].shape)
print("Attention mask shape:", tokens['attention_mask'].shape)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Input IDs shape: torch.Size([900, 8])
Attention mask shape: torch.Size([900, 8])


In [ ]:
#Create Dataset and DataLoader
import torch
from torch.utils.data import Dataset, DataLoader, random_split

# Step 5a: Define a custom dataset
class NameGenderDataset(Dataset):
    def __init__(self, tokens, labels):
        self.input_ids = tokens['input_ids']
        self.attention_mask = tokens['attention_mask']
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Step 5b: Create dataset object
dataset = NameGenderDataset(tokens, df['label'])

# Step 5c: Train-validation split (80/20)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Step 5d: Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

print(f"✅ DataLoader preparation complete — {train_size} train samples, {val_size} val samples")


✅ Dataset ready — 720 train samples, 180 val samples


In [ ]:
#Fine-Tune BERT for Gender Classification
from transformers import BertForSequenceClassification
from torch.optim import AdamW
from torch.nn import functional as F
from tqdm import tqdm

# Step 6a: Load BERT model for binary classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Step 6b: Define optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Step 6c: Training loop
epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()

    acc = correct / len(train_loader.dataset)
    print(f"✅ Epoch {epoch+1} — Loss: {total_loss:.4f}, Training Accuracy: {acc:.4f}")


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1: 100%|██████████| 45/45 [00:03<00:00, 11.47it/s]


✅ Epoch 1 — Loss: 18.4678, Train Accuracy: 0.8194


Epoch 2: 100%|██████████| 45/45 [00:02<00:00, 15.29it/s]


✅ Epoch 2 — Loss: 3.8637, Train Accuracy: 0.9806


Epoch 3: 100%|██████████| 45/45 [00:02<00:00, 15.16it/s]

✅ Epoch 3 — Loss: 3.7835, Train Accuracy: 0.9750


In [ ]:
# Step 7: Evaluate on validation set

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

val_accuracy = correct / total
print(f"✅ Model Validation Accuracy: {val_accuracy:.4f}")


✅ Validation Accuracy: 0.9556


In [ ]:
import os

# Create directory to save model
os.makedirs("bert_gender_classifier", exist_ok=True)

# Save model and tokenizer
model.save_pretrained("bert_gender_classifier")
tokenizer.save_pretrained("bert_gender_classifier")

print("✅ Model and tokenizer saved in /content/bert_gender_classifier")


✅ Model and tokenizer saved in /content/gender_bert_model


# **Gender Classification – Part 2: Serving & MLOps (rebuilt)**

This section replaces the original serving code, which trained and deployed a throwaway RandomForest on 10 hardcoded sample names. **The fine-tuned BERT model saved above is now the single deployed model**, served consistently across Flask, Docker, Kubernetes and MLflow.

Fixes: one canonical model and one `app.py`; paths resolved relative to the app file; full input validation (HTTP 400 on bad input); the API is **demonstrated live in this notebook** (no stale ngrok URL); MLflow uses a SQLite backend.

## 1. Project structure + place the trained model

In [ ]:
import os, shutil
PROJECT="/content/gender_project"; API=os.path.join(PROJECT,"gender_api")
for d in [API, PROJECT+"/k8s", PROJECT+"/tests", PROJECT+"/jenkins", PROJECT+"/docker"]:
    os.makedirs(d, exist_ok=True)
# Copy the fine-tuned BERT model next to the API (the model dir saved earlier)
dst=os.path.join(API,"bert_gender_classifier")
if os.path.exists(dst): shutil.rmtree(dst)
shutil.copytree("/content/bert_gender_classifier", dst)
print("Model placed at", dst, "->", os.listdir(dst)[:5])


## 2. REST API (Flask) — single BERT model, validated

In [ ]:
%%writefile /content/gender_project/gender_api/app.py
"""
Gender Classification REST API (BERT)
-------------------------------------
Serves the fine-tuned BERT model saved in ./bert_gender_classifier.

Fixes applied vs. the original notebook:
  * ONE model is served: the fine-tuned BERT classifier. The throwaway
    RandomForest-on-10-sample-names model and the duplicate app.py / Gradio
    variants are removed, so there is no ambiguity about what is deployed.
  * Model path is resolved relative to this file (works in Colab/Docker/K8s).
  * Heavy imports (torch/transformers) are lazy, so the module can be imported
    and unit-tested without loading the full model.
  * Input is validated: name must be a non-empty string; returns 400 otherwise.
  * Returns the predicted label AND a confidence score (softmax probability).
"""

import os
import logging

from flask import Flask, request, jsonify

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.environ.get("GENDER_MODEL_DIR", os.path.join(BASE_DIR, "bert_gender_classifier"))
LABELS = {0: "male", 1: "female"}

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("gender-api")

# Pluggable predictor: predict_fn(list[str]) -> list[(label:str, prob:float)]
# Left as None until first use so the module imports without torch.
PREDICT_FN = None


def _load_real_predictor():
    import torch
    from transformers import BertTokenizer, BertForSequenceClassification

    tokenizer = BertTokenizer.from_pretrained(MODEL_DIR)
    model = BertForSequenceClassification.from_pretrained(MODEL_DIR)
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    log.info("Loaded BERT model from %s on %s", MODEL_DIR, device)

    def predict_fn(names):
        enc = tokenizer(names, return_tensors="pt", padding=True, truncation=True)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=1)
            idx = torch.argmax(probs, dim=1)
        out = []
        for i, p in zip(idx.tolist(), probs.tolist()):
            out.append((LABELS[i], float(p[i])))
        return out

    return predict_fn


def get_predictor():
    global PREDICT_FN
    if PREDICT_FN is None:
        PREDICT_FN = _load_real_predictor()
    return PREDICT_FN


app = Flask(__name__)


def _extract_names(payload):
    """Accept either {"name": "X"} or {"names": ["X", "Y"]}. Validate types."""
    if not isinstance(payload, dict):
        raise ValueError("Request body must be a JSON object.")
    if "names" in payload:
        names = payload["names"]
        if not isinstance(names, list) or not names:
            raise ValueError("'names' must be a non-empty list of strings.")
    elif "name" in payload:
        names = [payload["name"]]
    else:
        raise ValueError("Provide a 'name' (string) or 'names' (list of strings).")
    for n in names:
        if not isinstance(n, str) or not n.strip():
            raise ValueError("Each name must be a non-empty string.")
    return [n.strip() for n in names]


@app.route("/", methods=["GET"])
def index():
    return jsonify({
        "service": "Gender Classification API (BERT)",
        "status": "running",
        "endpoints": {"/health": "GET", "/predict": "POST (JSON)"},
        "usage": {"name": "string"},
    })


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy"}), 200


@app.route("/predict", methods=["POST"])
def predict():
    if not request.is_json:
        return jsonify({"error": "Content-Type must be application/json."}), 415
    try:
        names = _extract_names(request.get_json(silent=True))
        results = get_predictor()(names)
        payload = [
            {"name": n, "predicted_gender": label, "confidence": round(prob, 4)}
            for n, (label, prob) in zip(names, results)
        ]
        return jsonify(payload[0] if len(payload) == 1 else {"results": payload}), 200
    except ValueError as ve:
        return jsonify({"error": str(ve)}), 400
    except Exception as exc:  # noqa: BLE001
        log.exception("Prediction failed")
        return jsonify({"error": "Internal server error.", "detail": str(exc)}), 500


if __name__ == "__main__":
    port = int(os.environ.get("PORT", "8000"))
    app.run(host="0.0.0.0", port=port)


In [ ]:
%%writefile /content/gender_project/gender_api/requirements.txt
flask==3.0.3
gunicorn==21.2.0
torch==2.2.2
transformers==4.40.1


## 3. Live API demonstration inside Colab
We load the real BERT model and call the API with Flask's test client — real inference, plus a bad request that must return `400`.

In [ ]:
import sys, importlib
sys.path.insert(0, "/content/gender_project/gender_api")
import app as gender_app; importlib.reload(gender_app)
gender_app.app.config["TESTING"] = True
client = gender_app.app.test_client()

print("HEALTH:", client.get("/health").get_json())
for nm in ["Priya", "Rahul", "Anjali", "John"]:
    print(f"{nm:8} ->", client.post("/predict", json={"name": nm}).get_json())
print("BATCH  ->", client.post("/predict", json={"names": ["Sneha","Amit"]}).get_json())
r = client.post("/predict", json={})           # missing name
print("MISSING-> status", r.status_code, r.get_json())


## 4. Containerization (Docker)

In [ ]:
%%writefile /content/gender_project/gender_api/Dockerfile
FROM python:3.11-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 PORT=8000
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
COPY bert_gender_classifier/ ./bert_gender_classifier/
RUN useradd -m appuser && chown -R appuser:appuser /app
USER appuser
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s --retries=3 \
  CMD python -c "import urllib.request,sys; sys.exit(0) if urllib.request.urlopen('http://localhost:8000/health').status==200 else sys.exit(1)"
CMD ["gunicorn","--bind","0.0.0.0:8000","--workers","1","--timeout","120","app:app"]


In [ ]:
%%writefile /content/gender_project/docker/build_and_test_docker.sh
#!/usr/bin/env bash
set -euo pipefail
IMAGE="gender-api:latest"
docker build -t "$IMAGE" -f gender_api/Dockerfile gender_api
docker run -d --rm -p 8000:8000 --name gender-api "$IMAGE"
sleep 25
curl -s http://localhost:8000/health; echo
curl -s -X POST http://localhost:8000/predict -H 'Content-Type: application/json' -d '{"name":"Priya"}'; echo
docker stop gender-api


In [ ]:
print("""Run on a Docker host and screenshot the output:
  cd gender_project
  bash docker/build_and_test_docker.sh
""")

## 5. Kubernetes deployment (labels matched across Deployment/Service)

In [ ]:
%%writefile /content/gender_project/k8s/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: gender-api-deployment
  labels:
    app: gender-api
spec:
  replicas: 1
  selector:
    matchLabels:
      app: gender-api
  template:
    metadata:
      labels:
        app: gender-api
    spec:
      containers:
        - name: gender-api
          image: gender-api:latest
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8000
          resources:
            requests:
              cpu: "250m"
              memory: "1Gi"
            limits:
              cpu: "1000m"
              memory: "2Gi"
          readinessProbe:
            httpGet: { path: /health, port: 8000 }
            initialDelaySeconds: 20
            periodSeconds: 10
          livenessProbe:
            httpGet: { path: /health, port: 8000 }
            initialDelaySeconds: 30
            periodSeconds: 20


In [ ]:
%%writefile /content/gender_project/k8s/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: gender-api-service
spec:
  type: NodePort
  selector:
    app: gender-api
  ports:
    - protocol: TCP
      port: 80
      targetPort: 8000
      nodePort: 30037


In [ ]:
import yaml
dep=yaml.safe_load(open("/content/gender_project/k8s/deployment.yaml"))
svc=yaml.safe_load(open("/content/gender_project/k8s/service.yaml"))
assert dep["spec"]["selector"]["matchLabels"]["app"]==dep["spec"]["template"]["metadata"]["labels"]["app"]==svc["spec"]["selector"]["app"]
print("K8s YAML valid; labels match ->", svc["spec"]["selector"]["app"])
print("""Deploy locally (minikube) and screenshot `kubectl get pods`:
  kubectl apply -f k8s/deployment.yaml
  kubectl apply -f k8s/service.yaml""")


## 6. Experiment tracking (MLflow, SQLite backend)

In [ ]:
!pip install -q mlflow

In [ ]:
import mlflow, mlflow.pytorch
mlflow.set_tracking_uri("sqlite:////content/mlflow.db")   # file:// store is retired in MLflow 3.x
mlflow.set_experiment("GenderClassificationExp")
with mlflow.start_run(run_name="bert-gender-classification"):
    mlflow.log_param("base_model", "bert-base-uncased")
    mlflow.log_param("epochs", 3)
    mlflow.log_param("batch_size", 16)
    mlflow.log_metric("val_accuracy", float(val_accuracy))   # from the eval cell above
    mlflow.pytorch.log_model(model, name="bert_gender_model")
    print("Logged BERT run to MLflow. val_accuracy =", round(float(val_accuracy),4))
print("View with:  mlflow ui --backend-store-uri sqlite:///mlflow.db")


## 7. CI/CD (Jenkins) + tests

In [ ]:
%%writefile /content/gender_project/tests/test_api.py
"""API tests for the gender service. Uses a stub predictor so the suite runs
without torch/transformers in CI's fast path; the live model is exercised in
the notebook and the container smoke test."""
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "gender_api"))
import pytest
import app as gender_app


@pytest.fixture
def client():
    # Inject a deterministic stub predictor (no torch needed).
    gender_app.PREDICT_FN = lambda names: [("female", 0.97) for _ in names]
    gender_app.app.config["TESTING"] = True
    with gender_app.app.test_client() as c:
        yield c


def test_health(client):
    assert client.get("/health").status_code == 200


def test_predict_single(client):
    r = client.post("/predict", json={"name": "Priya"})
    assert r.status_code == 200
    body = r.get_json()
    assert body["predicted_gender"] == "female"
    assert 0.0 <= body["confidence"] <= 1.0


def test_predict_batch(client):
    r = client.post("/predict", json={"names": ["Priya", "Rahul"]})
    assert r.status_code == 200
    assert len(r.get_json()["results"]) == 2


def test_missing_name_400(client):
    assert client.post("/predict", json={}).status_code == 400


def test_empty_name_400(client):
    assert client.post("/predict", json={"name": "  "}).status_code == 400


def test_non_json_415(client):
    assert client.post("/predict", data="x").status_code == 415


In [ ]:
%%writefile /content/gender_project/jenkins/Jenkinsfile
pipeline {
    agent any
    environment { IMAGE = "gender-api:${env.BUILD_NUMBER}" }
    stages {
        stage('Checkout') { steps { checkout scm } }
        stage('Setup & Test') {
            steps {
                sh '''
                    python3 -m venv .venv
                    . .venv/bin/activate
                    pip install --upgrade pip
                    pip install flask pytest
                    pytest -q tests/
                '''
            }
        }
        stage('Build Image') { steps { sh 'docker build -t $IMAGE -f gender_api/Dockerfile gender_api' } }
        stage('Smoke Test') {
            steps {
                sh '''
                    docker run -d --rm -p 8000:8000 --name gender-ci $IMAGE
                    sleep 25
                    curl -sf http://localhost:8000/health
                    docker stop gender-ci
                '''
            }
        }
    }
    post { always { cleanWs() } }
}


In [ ]:
!cd /content/gender_project && pip -q install pytest && python -m pytest -q tests/

## 8. Package for GitHub / submission

In [ ]:
%%writefile /content/gender_project/README.md
# Gender Classification — BERT (Classification Module)

Fine-tuned `bert-base-uncased` predicts gender from a user's name, served via a
Flask REST API, containerized with Docker, deployable to Kubernetes, tracked
with MLflow, and built via Jenkins CI/CD.

## Structure
```
gender_api/   app.py, requirements.txt, Dockerfile, bert_gender_classifier/
k8s/          deployment.yaml, service.yaml
tests/        test_api.py
jenkins/      Jenkinsfile
docker/       build_and_test_docker.sh
notebooks/    Gender_Classification_Model.ipynb
```

## Run locally
```bash
cd gender_api && pip install -r requirements.txt && python app.py
curl -X POST http://localhost:8000/predict -H 'Content-Type: application/json' -d '{"name":"Priya"}'
# -> {"name":"Priya","predicted_gender":"female","confidence":0.97}
```
Batch: `{"names":["Priya","Rahul"]}`. Invalid input returns HTTP 400.

## Docker / K8s / Jenkins / MLflow
See `docker/build_and_test_docker.sh`, `k8s/*.yaml`, `jenkins/Jenkinsfile`.
MLflow uses a SQLite backend (`sqlite:///mlflow.db`); view with `mlflow ui`.


In [ ]:
import shutil
shutil.make_archive("/content/gender_project_submission","zip","/content/gender_project")
from google.colab import files
files.download("/content/gender_project_submission.zip")
